In [1]:
# 分解调试 naive_converse：观察每步 history_g 和 history_e
# 请在 UoT 项目根目录下运行 (code/Reasoning/external/UoT)

import sys
import os
import json

# 连接 vLLM 必须的环境变量（须在 import src.uot 之前设置）
# 注意：QWEN_IP 必须带协议；若 vLLM 在别的机器/端口，请改下面两行
os.environ["QWEN_API_KEY"] = "EMPTY"
os.environ["QWEN_IP"] = "http://127.0.0.1"   # 不能写成 "127.0.0.1"，否则 base_url 缺协议会报错
os.environ["QWEN_PORT"] = "4321"

# 若之前已 import 过 src.uot.models，用当前 env 重新创建 vLLM 客户端，避免旧连接被缓存
if "src.uot.models" in sys.modules:
    from openai import OpenAI
    ip, port = os.environ.get("QWEN_IP", ""), os.environ.get("QWEN_PORT", "")
    base_url = f"{ip}:{port}/v1" if ip and port else ""
    if base_url:
        sys.modules["src.uot.models"].qwen_api_client = OpenAI(
            base_url=base_url,
            api_key=os.environ.get("QWEN_API_KEY", "EMPTY"),
        )
        print("已用当前 env 重新创建 qwen_api_client, base_url =", base_url)

# 项目根目录：优先使用含 run.py 的 UoT 目录
_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, "run.py")):
    UOT_ROOT = _cwd
else:
    for cand in [os.path.join(_cwd, "UoT"), os.path.join(_cwd, "external", "UoT"), os.path.join(_cwd, "code", "Reasoning", "external", "UoT")]:
        if os.path.isfile(os.path.join(cand, "run.py")):
            UOT_ROOT = cand
            break
    else:
        UOT_ROOT = _cwd
if UOT_ROOT not in sys.path:
    sys.path.insert(0, UOT_ROOT)
os.chdir(UOT_ROOT)
print("CWD:", os.getcwd())

CWD: /hpc2hdd/home/mpeng885/jiawenzhang/code/Reasoning/external/UoT


In [2]:
# 检查 vLLM 是否可达（可选：先跑本 cell，确认能连再跑后面的 Guesser/Examiner）
def check_vllm_connection():
    import urllib.request
    import urllib.error
    ip, port = os.environ.get("QWEN_IP", ""), os.environ.get("QWEN_PORT", "")
    if not ip or not port:
        print("未设置 QWEN_IP / QWEN_PORT")
        return False
    base = f"{ip}:{port}"
    url = f"{base}/v1/models"  # OpenAI 兼容接口
    try:
        req = urllib.request.Request(url, method="GET")
        with urllib.request.urlopen(req, timeout=5) as r:
            print("vLLM 连接正常:", url, "->", r.status)
            return True
    except urllib.error.URLError as e:
        print("vLLM 连接失败:", url)
        print("  错误:", e.reason if hasattr(e, "reason") else e)
        print("  请确认: 1) vLLM 服务已启动  2) QWEN_IP/QWEN_PORT 正确（本机用 http://127.0.0.1:4321）")
        return False
    except Exception as e:
        print("vLLM 连接异常:", e)
        return False

check_vllm_connection()

vLLM 连接正常: http://127.0.0.1:4321/v1/models -> 200


True

In [3]:
# 构建 task（与 run.py 一致的参数）
import argparse
from src.uot.tasks import get_task

def make_args(
    task="20q",
    dataset="common",
    guesser_model="qwen_4b",
    examiner_model="qwen_4b",
    max_turn=20,
    open_set_size=-1,
    task_start_index=-1,
    task_end_index=-1,
):
    p = argparse.ArgumentParser()
    p.add_argument("--guesser_model", type=str, default=guesser_model)
    p.add_argument("--examiner_model", type=str, default=examiner_model)
    p.add_argument("--task", type=str, default=task)
    p.add_argument("--dataset", type=str, default=dataset)
    p.add_argument("--max_turn", type=int, default=max_turn)
    p.add_argument("--open_set_size", type=int, default=open_set_size)
    p.add_argument("--task_start_index", type=int, default=task_start_index)
    p.add_argument("--task_end_index", type=int, default=task_end_index)
    p.add_argument("--temperature", type=float, default=0)
    p.add_argument("--naive_run", action="store_true", default=True)
    p.add_argument("--inform", action="store_true", default=False)
    p.add_argument("--reward_lambda", type=float, default=0.4)
    p.add_argument("--n_extend_layers", type=int, default=3)
    p.add_argument("--n_potential_actions", type=int, default=3)
    p.add_argument("--n_pruned_nodes", type=float, default=0)
    p.add_argument("--add_info", type=str, default="")
    p.add_argument("--expected_action_tokens", type=int, default=500)
    p.add_argument("--expected_target_tokens", type=int, default=20)
    p.add_argument("--none_acc_reward", action="store_true", default=False)
    p.add_argument("--expected_reward_method", type=str, default="avg")
    p.add_argument("--size_to_renew", type=int, default=-1)
    p.add_argument("--n_pre_ask", type=int, default=0)
    return p.parse_args([])

args = make_args()
task = get_task(args)
# 选第几条数据调试
i = 0
item = task.data[i]["target"]
print("task.data[i]:", task.data[i])
print("target item:", item)

QWEN_API_KEY: ****MPTY
task.data[i]: {'target': 'Meerkat'}
target item: Meerkat


In [4]:
def print_histories(history_g, history_e, step_name=""):
    """打印当前 step 的 history_g 和 history_e，便于观察每步产生的内容。"""
    print("=" * 60, step_name, "=" * 60)
    print("\n--- history_g (Guesser 看到的对话) ---")
    for idx, msg in enumerate(history_g):
        role = msg.get("role", "?")
        content = msg.get("content", "")
        preview = content
        # preview = (content[:200] + "..." if len(content) > 200 else content).replace("\n", " ")
        print(f"  [{idx}] {role}: {preview}")
    print("\n--- history_e (Examiner 看到的对话) ---")
    for idx, msg in enumerate(history_e):
        role = msg.get("role", "?")
        content = msg.get("content", "")
        preview = content
        # preview = (content[:200] + "..." if len(content) > 200 else content).replace("\n", " ")
        print(f"  [{idx}] {role}: {preview}")
    print()

## Step 0：初始化 history_g 和 history_e（不调用模型）

与 `naive_converse` 开头一致：根据是否有 `self_repo` 初始化 Guesser 的 system；根据是否 `free_answer` 初始化 Examiner 的 system。

In [5]:
# Step 0: 初始化（对应 naive_converse 第 449-483 行）
item = task.data[i]["target"]
target_decl = task.prompts.target_declaration.format(target=item)
print("target_decl:", target_decl)

if "self_repo" in task.data[i]:
    guesser_prologue = (
        task.prompts.guesser_prologue_FA if task.free_answer
        else task.prompts.guesser_prologue
    )
    history_g = [{"role": "system", "content": guesser_prologue.format(repo=task.data[i]["self_repo"])}]
    print("Self-report:", task.data[i]["self_repo"])
else:
    history_g = [{"role": "system", "content": task.prompts.guesser_prologue}]

if not task.free_answer:
    history_e = [{"role": "system", "content": task.prompts.examiner_prologue.format(item=item)}]
else:
    history_e = [{
        "role": "system",
        "content": task.prompts.simulator_prologue.format(
            item=item,
            conv_hist=task.data[i]["conv_hist"]
        )
    }]

print_histories(history_g, history_e, "Step 0 初始化后")

target_decl: Target is 'Meerkat'
============================================================ Step 0 初始化后 ============================================================

--- history_g (Guesser 看到的对话) ---
  [0] system: Let us play the game of 20 questions. I am impersonating the thing, X. You will ask me up to 20 questions which start with 'Is X' and can only be answered by yes or no, and I will answer each one truthfully based on being X.
Let us begin. Ask me the first question.


--- history_e (Examiner 看到的对话) ---
  [0] system: Let us play the game of 20 questions. You are the answerer and I am guesser. X is 'Meerkat'. 
I will ask you up to 20 questions and you should answer each one truthfully based on being X. 
If I guess correctly what X is, answer me "You guessed it. X is 'Meerkat'."
Note that never directly tell me what X is all the time.
Let us begin. Here is my first question.




## Step 1：第一次 Guesser 响应

调用 `get_guesser_naive_response(task, history_g, count+1)`（count=0），得到 Bot 2 的回复，然后：
- `history_g.append({'role': 'assistant', 'content': bot1_response_text})`
- `history_e.append({'role': 'user', 'content': bot1_response_text})`

In [6]:
# Step 1: 第一次 Guesser（对应 naive_converse 第 486-499 行）
from src.uot.method import get_guesser_naive_response

count = 0
bot1_response_text, bot1_thinking_text = get_guesser_naive_response(task, history_g, count + 1)
print("Bot 2 (Guesser):", bot1_response_text)
if bot1_thinking_text:
    print("(thinking 已省略)")

history_g.append({"role": "assistant", "content": bot1_response_text})
history_e.append({"role": "user", "content": bot1_response_text})

print_histories(history_g, history_e, "Step 1 第一次 Guesser 回复后")

Bot 2 (Guesser): Is X a living thing?
(thinking 已省略)
============================================================ Step 1 第一次 Guesser 回复后 ============================================================

--- history_g (Guesser 看到的对话) ---
  [0] system: Let us play the game of 20 questions. I am impersonating the thing, X. You will ask me up to 20 questions which start with 'Is X' and can only be answered by yes or no, and I will answer each one truthfully based on being X.
Let us begin. Ask me the first question.

  [1] assistant: Is X a living thing?

--- history_e (Examiner 看到的对话) ---
  [0] system: Let us play the game of 20 questions. You are the answerer and I am guesser. X is 'Meerkat'. 
I will ask you up to 20 questions and you should answer each one truthfully based on being X. 
If I guess correctly what X is, answer me "You guessed it. X is 'Meerkat'."
Note that never directly tell me what X is all the time.
Let us begin. Here is my first question.

  [1] user: Is X a living thing?



## Step 2：Examiner 响应

调用 `get_examiner_response(task, history_e)`，得到 Bot 1 的回复，然后：
- `history_e.append({'role': 'assistant', 'content': bot2_response_text})`
- `history_g.append({'role': 'user', 'content': bot2_response_text})`

并检查是否猜中（`is_winning_response`）或达到 `max_turn`。

In [7]:
# Step 2: Examiner 第一次回复（对应 naive_converse while 内 500-514 行）
from src.uot.method import get_examiner_response, is_winning_response

_, bot2_response_text, bot2_thinking_text = get_examiner_response(task, history_e)
print("Bot 1 (Examiner):", bot2_response_text)

history_e.append({"role": "assistant", "content": bot2_response_text})
history_g.append({"role": "user", "content": bot2_response_text})

print_histories(history_g, history_e, "Step 2 Examiner 第一次回复后")

# 检查是否结束
print("is_winning_response?", is_winning_response(bot2_response_text))
print("count =", count, ", max_turn =", task.max_turn)

Bot 1 (Examiner): Yes.
============================================================ Step 2 Examiner 第一次回复后 ============================================================

--- history_g (Guesser 看到的对话) ---
  [0] system: Let us play the game of 20 questions. I am impersonating the thing, X. You will ask me up to 20 questions which start with 'Is X' and can only be answered by yes or no, and I will answer each one truthfully based on being X.
Let us begin. Ask me the first question.

  [1] assistant: Is X a living thing?
  [2] user: Yes.

--- history_e (Examiner 看到的对话) ---
  [0] system: Let us play the game of 20 questions. You are the answerer and I am guesser. X is 'Meerkat'. 
I will ask you up to 20 questions and you should answer each one truthfully based on being X. 
If I guess correctly what X is, answer me "You guessed it. X is 'Meerkat'."
Note that never directly tell me what X is all the time.
Let us begin. Here is my first question.

  [1] user: Is X a living thing?
  [2] assistan

## Step 3：继续下一轮（count+=1 后再次 Guesser）

若未猜中且未到 max_turn，则 count+=1，再调用 Guesser，再更新 history_g / history_e。

In [ ]:
# Step 3: 下一轮 Guesser（对应 naive_converse 517-535 行）
if not is_winning_response(bot2_response_text) and count + 1 < task.max_turn:
    count += 1
    print("------", count, "-------------")
    bot1_response_text, bot1_thinking_text = get_guesser_naive_response(task, history_g, count + 1)
    print("Bot 2:", bot1_response_text)
    history_g.append({"role": "assistant", "content": bot1_response_text})
    history_e.append({"role": "user", "content": bot1_response_text})
    print_histories(history_g, history_e, f"Step 3 第 {count+1} 轮 Guesser 回复后")
else:
    print("(已猜中或达到 max_turn，不再继续)")

---

## 可选：不调模型，用 Mock 数据完整走一遍流程

下面用假回复模拟一轮对话，只展示**每步** `history_g` 和 `history_e` 的**结构**和**内容**，不调用真实模型。便于理解 `naive_converse` 的每一步如何修改这两条 history。

In [ ]:
# Mock 版：分解 naive_converse，每步打印 history_g / history_e（不调模型）
def naive_converse_mock_debug(task, i, max_steps=3):
    """与 naive_converse 逻辑一致，但用假回复，并在每步打印 history_g / history_e。"""
    item = task.data[i]["target"]
    target_decl = task.prompts.target_declaration.format(target=item)

    if "self_repo" in task.data[i]:
        guesser_prologue = (
            task.prompts.guesser_prologue_FA if task.free_answer
            else task.prompts.guesser_prologue
        )
        history_g = [{"role": "system", "content": guesser_prologue.format(repo=task.data[i]["self_repo"])}]
    else:
        history_g = [{"role": "system", "content": task.prompts.guesser_prologue}]

    if not task.free_answer:
        history_e = [{"role": "system", "content": task.prompts.examiner_prologue.format(item=item)}]
    else:
        history_e = [{
            "role": "system",
            "content": task.prompts.simulator_prologue.format(
                item=item,
                conv_hist=task.data[i].get("conv_hist", "")
            )
        }]

    print_histories(history_g, history_e, "[Mock] Step 0 初始化")

    count = 0
    # 第一轮 Guesser
    bot1_response_text = "[Mock] Is X a living thing?"
    history_g.append({"role": "assistant", "content": bot1_response_text})
    history_e.append({"role": "user", "content": bot1_response_text})
    print_histories(history_g, history_e, "[Mock] Step 1 第一次 Guesser 回复后")

    for step in range(max_steps - 1):
        bot2_response_text = "[Mock] No, X is not a living thing." if step == 0 else "[Mock] You guessed it. X is '" + item + "'."
        history_e.append({"role": "assistant", "content": bot2_response_text})
        history_g.append({"role": "user", "content": bot2_response_text})
        print_histories(history_g, history_e, f"[Mock] Step 2.{step+1} Examiner 回复后")

        if "You guessed it" in bot2_response_text:
            print("-> 猜中，结束")
            break
        count += 1
        if count >= task.max_turn:
            print("-> 达到 max_turn，结束")
            break

        bot1_response_text = f"[Mock] Is X {item}?"
        history_g.append({"role": "assistant", "content": bot1_response_text})
        history_e.append({"role": "user", "content": bot1_response_text})
        print_histories(history_g, history_e, f"[Mock] Step 3.{step+1} 下一轮 Guesser 回复后")

    return history_g, history_e

# 只跑 3 步，看前几轮结构
naive_converse_mock_debug(task, i, max_steps=3)